# mROSE sequence generation demo

This notebook provides a GitHub-friendly walkthrough for checking the mROSE installation, verifying released checkpoints and running compact sequence-generation demos.

It is designed to run from a cloned copy of the repository. The checked-in version runs the compact CDS demo so GitHub and Read the Docs can display real generation output; change `RUN_DEMO` to `False` before re-running if you only want the setup checks.

<p align="center">
  <img src="../docs/assets/mrose-icon.png" alt="mROSE icon" width="160">
</p>

<p align="center">
  <img src="../docs/assets/mrose-figure1.png" alt="mROSE workflow" width="900">
</p>

## 1. Locate the repository

Run this notebook from `notebooks/` inside a local clone of `xiaoshaziydy/mrose`, or from the repository root. The following cell detects the project root and adds it to `sys.path`.

In [1]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
if (cwd / "mrose").is_dir():
    PROJECT_ROOT = cwd
elif cwd.name == "notebooks" and (cwd.parent / "mrose").is_dir():
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: /yangwenyi/mrose-rtd-run


## 2. Installation notes

Clone with Git LFS enabled so the model checkpoints are downloaded as real `.pth` files rather than small pointer files.

```bash
git lfs install
git clone https://github.com/xiaoshaziydy/mrose.git
cd mrose
git lfs pull
conda env create -f environment.yml
conda activate mROSE
```

Alternatively, install the core pip dependencies:

```bash
pip install --extra-index-url https://download.pytorch.org/whl/cu118 -r requirements.txt
```

## 3. Dependency and checkpoint status

In [2]:
import importlib.util

required_modules = ["numpy", "pandas", "torch", "Bio", "sklearn", "scipy", "tqdm"]
optional_modules = ["RNA", "ViennaRNA"]

def module_available(name: str) -> bool:
    return importlib.util.find_spec(name) is not None

print("Dependency check")
for name in required_modules:
    print(f"  {name:10s}: {'OK' if module_available(name) else 'missing'}")
print("Optional RNA folding bindings")
for name in optional_modules:
    print(f"  {name:10s}: {'OK' if module_available(name) else 'missing'}")

Dependency check
  numpy     : OK
  pandas    : OK
  torch     : OK
  Bio       : OK
  sklearn   : OK
  scipy     : OK
  tqdm      : OK
Optional RNA folding bindings
  RNA       : OK
  ViennaRNA : OK


In [3]:
checkpoints = {
    "5utr": PROJECT_ROOT / "generation" / "5utr" / "Model.pth",
    "cds": PROJECT_ROOT / "generation" / "cds" / "Model.pth",
    "3utr": PROJECT_ROOT / "generation" / "3utr" / "Model.pth",
}

print("Checkpoint check")
for region, path in checkpoints.items():
    if path.exists():
        size_mb = path.stat().st_size / (1024 * 1024)
        print(f"  {region:4s}: {path.relative_to(PROJECT_ROOT)} ({size_mb:.1f} MB)")
    else:
        print(f"  {region:4s}: missing at {path.relative_to(PROJECT_ROOT)}")

Checkpoint check
  5utr: generation/5utr/Model.pth (436.5 MB)
  cds : generation/cds/Model.pth (274.2 MB)
  3utr: generation/3utr/Model.pth (287.1 MB)


## 4. Quick import check

In [4]:
try:
    from mrose import (
        Model_5UTR,
        Model_CDS,
        Model_3UTR,
        GlobalmRNAMasterModel,
        compute_features_5utr,
        compute_features_3utr,
    )

    print("mROSE imports OK")
    print("5UTR feature length:", len(compute_features_5utr("ATGCGTACGTAG")))
    print("3UTR feature length:", len(compute_features_3utr("ATGCGTACGTAG")))
    print("Core classes:", Model_5UTR.__name__, Model_CDS.__name__, Model_3UTR.__name__, GlobalmRNAMasterModel.__name__)
except Exception as exc:
    print("Import check failed:", type(exc).__name__, exc)
    print("Install the environment first, then rerun this cell.")

mROSE imports OK
5UTR feature length: 27
3UTR feature length: 23
Core classes: Model_5UTR Model_CDS Model_3UTR GlobalmRNAMasterModel


## 5. Print ready-to-run generation commands

The repository includes a small launcher that prints commands for all supported generation demos and checks local dependencies/checkpoints.

In [5]:
import subprocess

result = subprocess.run(
    [sys.executable, str(PROJECT_ROOT / "scripts" / "demo_generate_sequences.py")],
    cwd=PROJECT_ROOT,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    check=False,
)
print(result.stdout)

mROSE generation demo
Project root: /yangwenyi/mrose-rtd-run

Dependency check:
  numpy      OK
  pandas     OK
  torch      OK
  Bio        OK
  sklearn    OK
  scipy      OK
  tqdm       OK
  ViennaRNA  OK

Checkpoint check:
  5utr generation/5utr/Model.pth: OK (436.5 MB)
  cds  generation/cds/Model.pth: OK (274.2 MB)
  3utr generation/3utr/Model.pth: OK (287.1 MB)

Commands:

# 5' UTR
/root/miniconda3/envs/DiffRNA/bin/python /yangwenyi/mrose-rtd-run/generation/5utr/generate_5utr.py --checkpoint /yangwenyi/mrose-rtd-run/generation/5utr/Model.pth --input_fasta /yangwenyi/mrose-rtd-run/generation/examples/5utr_template.fasta --output_dir /yangwenyi/mrose-rtd-run/outputs/generation/5utr_demo --num_samples 20 --top_k 5 --device cpu --output_prefix demo_5utr

# CDS
/root/miniconda3/envs/DiffRNA/bin/python /yangwenyi/mrose-rtd-run/generation/cds/generate_cds.py --checkpoint /yangwenyi/mrose-rtd-run/generation/cds/Model.pth --input_fasta /yangwenyi/mrose-rtd-run/generation/examples/cds_temp

## 6. Run a compact generation demo

This cell runs the compact CDS demo by default. `cds` is the lightest choice because the demo disables MFE scoring with `--mfe_weight 0`; set `RUN_DEMO = False` if you only want to preview the notebook without inference.

In [6]:
RUN_DEMO = True
TASK = "cds"  # choose from: "5utr", "cds", "3utr", "all"

if RUN_DEMO:
    result = subprocess.run(
        [sys.executable, str(PROJECT_ROOT / "scripts" / "demo_generate_sequences.py"), "--run", TASK],
        cwd=PROJECT_ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=True,
    )
    print(result.stdout)
else:
    print("Demo execution skipped. Set RUN_DEMO = True to run inference.")

Input length: 117 nt
Using device: cpu
Using one checkpoint for both generation and prediction: /yangwenyi/mrose-rtd-run/generation/cds/Model.pth
Scoring formula: final_score = 0.65 * normalized(pred_value) + 0.2 * CAI + 0.05 * GC_window_score + 0 * MFE_score - internal_stop_penalty
MFE direction: lower (lower = more negative MFE is better)
Command-line model hyperparameters already match the checkpoint.
Loaded single checkpoint model for generation and prediction: embed_dim=192, hidden_dim=384, latent_dim=384, max_len=768, device=cpu
Generating chunk 1/1: 117 nt / 39 codons
All generated sequences have matched length: 117 nt
Deduplicated candidates: 20 -> 20
Predicting generated sequence values with the same checkpoint/model...
Calculating MFE with ViennaRNA...

MFE: 100%|██████████| 20/20 [00:00<00:00, 77.80it/s]
Saved all candidates: /yangwenyi/mrose-rtd-run/outputs/generation/cds_demo/all_generated_scored.csv
Saved Top5 CSV: /yangwenyi/mrose-rtd-run/outputs/generation/cds_demo/top1

## 7. Inspect generated outputs

After running a demo, generated files are written under `outputs/generation/`.

In [7]:
output_root = PROJECT_ROOT / "outputs" / "generation"
if output_root.exists():
    for path in sorted(output_root.rglob("*")):
        if path.is_file():
            print(path.relative_to(PROJECT_ROOT))
else:
    print("No generation outputs found yet.")

outputs/generation/cds_demo/all_generated_scored.csv
outputs/generation/cds_demo/top10_generated_scored.csv
outputs/generation/cds_demo/top10_generated_scored.fasta


## 8. Reproducibility checklist

- Use Git LFS before cloning or run `git lfs pull` after cloning.
- Verify released checkpoints with `shasum -a 256 -c MODEL_CHECKSUMS.sha256`.
- Keep full-scale datasets outside normal Git history.
- Use Zenodo or another archival repository for large public data releases.